
# 01. Prepare Data

고빈도 고객을 추출하고, 누수 없는 탭형 샘플과 LSTM 시퀀스 데이터를 생성합니다.


In [1]:

%matplotlib inline

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

ORDERS_PATH = 'data/orders.csv'
FEATURE_PATH_CANDIDATES = [
    'data/customer_features.csv',
    'data/script_job_3ea69affb682fff4480bec5a3efe5361_1.csv',
]
OUTPUT_DIR = Path('outputs')
HIGH_VALUE_QUANTILE = 0.8
DELAY_THRESHOLD = 15
SEQ_LEN = 5
VAL_RATIO = 0.15
TEST_RATIO = 0.15

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str((OUTPUT_DIR / '.mplconfig').resolve()))
(OUTPUT_DIR / '.mplconfig').mkdir(parents=True, exist_ok=True)


In [2]:
def print_section(title: str) -> None:
    # 구간 출력
    line = '=' * 50
    print(f'\n{line}\n{title}\n{line}')


def resolve_feature_path() -> Path:
    # 피처 경로 확인
    for candidate in FEATURE_PATH_CANDIDATES:
        path = Path(candidate)
        if path.exists():
            return path
    raise FileNotFoundError(f'피처 파일이 없습니다: {FEATURE_PATH_CANDIDATES}')


def require_columns(frame: pd.DataFrame, columns: list[str], context: str) -> None:
    # 컬럼 확인
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise ValueError(f'{context} 필수 컬럼이 없습니다: {missing}')


def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    # 데이터 로드
    print_section('[INFO] 데이터 로드')
    orders_path = Path(ORDERS_PATH)
    feature_path = resolve_feature_path()
    if not orders_path.exists():
        raise FileNotFoundError(f'orders 파일이 없습니다: {orders_path}')

    orders_header = pd.read_csv(orders_path, nrows=0)
    feature_header = pd.read_csv(feature_path, nrows=0)
    print(f'[INFO] orders 컬럼: {list(orders_header.columns)}')
    print(f'[INFO] feature 컬럼: {list(feature_header.columns)}')

    require_columns(orders_header, ['user_id', 'order_number', 'days_since_prior_order'], 'orders')
    require_columns(feature_header, ['user_id', 'total_orders'], 'feature')

    orders = pd.read_csv(
        orders_path,
        usecols=[column for column in ['user_id', 'order_number', 'days_since_prior_order', 'order_dow', 'order_hour_of_day'] if column in orders_header.columns],
        dtype={
            'user_id': 'int32',
            'order_number': 'int16',
            'days_since_prior_order': 'float32',
            'order_dow': 'float32',
            'order_hour_of_day': 'float32',
        },
    )
    features = pd.read_csv(
        feature_path,
        usecols=['user_id', 'total_orders'],
        dtype={'user_id': 'int32', 'total_orders': 'float32'},
    )

    orders['days_since_prior_order'] = orders['days_since_prior_order'].fillna(0.0)
    if 'order_dow' not in orders.columns:
        orders['order_dow'] = 0.0
    if 'order_hour_of_day' not in orders.columns:
        orders['order_hour_of_day'] = 0.0
    orders['order_dow'] = orders['order_dow'].fillna(0.0)
    orders['order_hour_of_day'] = orders['order_hour_of_day'].fillna(0.0)
    orders = orders.sort_values(['user_id', 'order_number']).reset_index(drop=True)
    return orders, features


def filter_high_value_orders(orders: pd.DataFrame, features: pd.DataFrame) -> pd.DataFrame:
    # 고객 필터링
    print_section('[INFO] 고빈도 고객 선택')
    threshold = float(features['total_orders'].quantile(HIGH_VALUE_QUANTILE))
    high_value_users = features.loc[features['total_orders'] >= threshold, 'user_id'].drop_duplicates()
    filtered = orders[orders['user_id'].isin(high_value_users)].copy()
    if filtered.empty:
        raise ValueError('고빈도 고객 주문 데이터가 없습니다.')
    print(f'[INFO] total_orders 80분위수: {threshold:.2f}')
    print(f'[INFO] 고빈도 고객 수: {high_value_users.nunique()}')
    print(f'[INFO] 주문 수: {len(filtered)}')
    return filtered


def build_samples_for_user(user_orders: pd.DataFrame) -> list[dict]:
    # 샘플 생성
    user_orders = user_orders.sort_values('order_number').reset_index(drop=True)
    if len(user_orders) < SEQ_LEN + 3:
        return []

    gaps = user_orders['days_since_prior_order'].to_numpy(dtype=np.float32)
    dows = user_orders['order_dow'].to_numpy(dtype=np.float32)
    hours = user_orders['order_hour_of_day'].to_numpy(dtype=np.float32)
    order_numbers = user_orders['order_number'].to_numpy(dtype=np.int32)
    rows = []

    for target_idx in range(SEQ_LEN, len(user_orders)):
        history = user_orders.iloc[:target_idx]
        positive_gaps = history.loc[history['days_since_prior_order'] > 0, 'days_since_prior_order'].to_numpy(dtype=np.float32)
        recent_gaps = positive_gaps[-3:]
        last_gap = float(gaps[target_idx - 1]) if target_idx >= 1 else 0.0
        prev_gap = float(gaps[target_idx - 2]) if target_idx >= 2 else 0.0
        total_gap_days = float(positive_gaps.sum()) if len(positive_gaps) else 0.0
        hist_total_orders = int(target_idx)
        rows.append({
            'user_id': int(user_orders.iloc[0]['user_id']),
            'target_order_number': int(order_numbers[target_idx]),
            'label': int(gaps[target_idx] > DELAY_THRESHOLD),
            'hist_total_orders': hist_total_orders,
            'hist_total_orders_log': float(np.log1p(hist_total_orders)),
            'hist_mean_gap': float(positive_gaps.mean()) if len(positive_gaps) else 0.0,
            'hist_std_gap': float(positive_gaps.std()) if len(positive_gaps) else 0.0,
            'hist_min_gap': float(positive_gaps.min()) if len(positive_gaps) else 0.0,
            'hist_max_gap': float(positive_gaps.max()) if len(positive_gaps) else 0.0,
            'hist_last_gap': last_gap,
            'hist_gap_mean_3': float(recent_gaps.mean()) if len(recent_gaps) else 0.0,
            'hist_gap_std_3': float(recent_gaps.std()) if len(recent_gaps) else 0.0,
            'hist_gap_change_1': float(last_gap - prev_gap),
            'hist_order_span_days': total_gap_days,
            'hist_order_frequency': float(hist_total_orders / max(total_gap_days, 1.0)),
            'hist_mean_dow': float(dows[:target_idx].mean()),
            'hist_mean_hour': float(hours[:target_idx].mean()),
            'hist_last_dow': float(dows[target_idx - 1]),
            'hist_last_hour': float(hours[target_idx - 1]),
        })
    return rows


def build_time_aware_samples(filtered_orders: pd.DataFrame) -> pd.DataFrame:
    # 누수 방지 처리
    print_section('[INFO] 시점 기준 샘플 생성')
    rows = []
    for _, user_orders in filtered_orders.groupby('user_id', sort=False):
        rows.extend(build_samples_for_user(user_orders))
    if not rows:
        raise ValueError('생성된 샘플이 없습니다.')
    samples = pd.DataFrame(rows)
    samples.insert(0, 'sample_id', np.arange(len(samples), dtype=np.int32))
    print(f'[INFO] 샘플 수: {len(samples)}')
    return samples


def assign_splits(samples: pd.DataFrame) -> pd.DataFrame:
    # 시점 기준 분할
    print_section('[INFO] train/val/test 분할')
    assigned = []
    for _, user_samples in samples.groupby('user_id', sort=False):
        user_samples = user_samples.sort_values('target_order_number').reset_index(drop=True)
        count = len(user_samples)
        if count < 3:
            continue
        test_count = max(1, int(np.ceil(count * TEST_RATIO)))
        val_count = max(1, int(np.ceil(count * VAL_RATIO)))
        train_count = count - test_count - val_count
        if train_count < 1:
            train_count, val_count, test_count = count - 2, 1, 1
        if train_count < 1:
            continue
        split = ['train'] * train_count + ['val'] * val_count + ['test'] * test_count
        user_samples = user_samples.iloc[:len(split)].copy()
        user_samples['split'] = split
        assigned.append(user_samples)
    if not assigned:
        raise ValueError('분할 가능한 샘플이 없습니다.')
    result = pd.concat(assigned, ignore_index=True)
    for split_name in ['train', 'val', 'test']:
        labels = result.loc[result['split'] == split_name, 'label'].to_numpy(dtype=np.int32)
        if len(np.unique(labels)) < 2:
            raise ValueError(f'{split_name} 라벨이 한 클래스만 있습니다.')
    print(result['split'].value_counts().to_string())
    return result


def build_sequence_arrays(filtered_orders: pd.DataFrame, samples: pd.DataFrame) -> tuple[dict, list[str]]:
    # 시퀀스 생성
    print_section('[INFO] LSTM 시퀀스 생성')
    lookup = {int(user_id): frame.sort_values('order_number').reset_index(drop=True) for user_id, frame in filtered_orders.groupby('user_id', sort=False)}
    feature_names = [
        'seq_gap',
        'seq_gap_change',
        'seq_order_norm',
        'seq_order_dow',
        'seq_order_hour',
        'static_hist_total_orders_log',
        'static_hist_mean_gap',
        'static_hist_gap_mean_3',
        'static_hist_order_frequency',
    ]
    arrays = {name: [] for name in ['x_train', 'x_val', 'x_test', 'y_train', 'y_val', 'y_test', 'test_sample_ids']}

    for row in samples.itertuples(index=False):
        user_orders = lookup.get(int(row.user_id))
        target_idx_arr = np.where(user_orders['order_number'].to_numpy() == int(row.target_order_number))[0]
        if len(target_idx_arr) == 0:
            continue
        target_idx = int(target_idx_arr[0])
        if target_idx < SEQ_LEN:
            continue
        window = user_orders.iloc[target_idx - SEQ_LEN:target_idx].copy()
        gaps = window['days_since_prior_order'].to_numpy(dtype=np.float32)
        gap_change = np.diff(np.concatenate(([gaps[0]], gaps))).astype(np.float32)
        order_norm = window['order_number'].to_numpy(dtype=np.float32) / max(float(window['order_number'].iloc[-1]), 1.0)
        dows = window['order_dow'].to_numpy(dtype=np.float32)
        hours = window['order_hour_of_day'].to_numpy(dtype=np.float32)
        static_vector = np.array([
            float(row.hist_total_orders_log),
            float(row.hist_mean_gap),
            float(row.hist_gap_mean_3),
            float(row.hist_order_frequency),
        ], dtype=np.float32)
        static_window = np.repeat(static_vector.reshape(1, -1), SEQ_LEN, axis=0)
        sequence = np.column_stack([gaps, gap_change, order_norm, dows, hours]).astype(np.float32)
        sequence = np.concatenate([sequence, static_window], axis=1)
        split_name = row.split
        arrays[f'x_{split_name}'].append(sequence)
        arrays[f'y_{split_name}'].append(int(row.label))
        if split_name == 'test':
            arrays['test_sample_ids'].append(int(row.sample_id))

    result = {}
    for split_name in ['train', 'val', 'test']:
        if not arrays[f'x_{split_name}']:
            raise ValueError(f'{split_name} 시퀀스가 없습니다.')
        result[f'X_{split_name}'] = np.stack(arrays[f'x_{split_name}']).astype(np.float32)
        result[f'y_{split_name}'] = np.asarray(arrays[f'y_{split_name}'], dtype=np.int32)
        print(f'[INFO] {split_name} shape: {result[f"X_{split_name}"].shape}')
    result['test_sample_ids'] = np.asarray(arrays['test_sample_ids'], dtype=np.int32)
    return result, feature_names


def save_outputs(samples: pd.DataFrame, sequence_data: dict, feature_names: list[str]) -> None:
    # 결과 저장
    print_section('[INFO] 전처리 결과 저장')
    samples.to_csv(OUTPUT_DIR / 'samples.csv', index=False)
    np.save(OUTPUT_DIR / 'X_train_seq.npy', sequence_data['X_train'])
    np.save(OUTPUT_DIR / 'X_val_seq.npy', sequence_data['X_val'])
    np.save(OUTPUT_DIR / 'X_test_seq.npy', sequence_data['X_test'])
    np.save(OUTPUT_DIR / 'y_train.npy', sequence_data['y_train'])
    np.save(OUTPUT_DIR / 'y_val.npy', sequence_data['y_val'])
    np.save(OUTPUT_DIR / 'y_test.npy', sequence_data['y_test'])
    np.save(OUTPUT_DIR / 'test_sample_ids.npy', sequence_data['test_sample_ids'])
    with open(OUTPUT_DIR / 'sequence_feature_names.json', 'w', encoding='utf-8') as file:
        json.dump({'feature_names': feature_names}, file, ensure_ascii=False, indent=2)
    with open(OUTPUT_DIR / 'tabular_feature_names.json', 'w', encoding='utf-8') as file:
        json.dump({'feature_names': [column for column in samples.columns if column.startswith('hist_')]}, file, ensure_ascii=False, indent=2)
    summary = {
        'sample_count': int(len(samples)),
        'user_count': int(samples['user_id'].nunique()),
        'split_counts': {key: int(value) for key, value in samples['split'].value_counts().to_dict().items()},
        'label_counts': {str(key): int(value) for key, value in samples['label'].value_counts().sort_index().to_dict().items()},
    }
    with open(OUTPUT_DIR / 'preprocess_summary.json', 'w', encoding='utf-8') as file:
        json.dump(summary, file, ensure_ascii=False, indent=2)
    print(f'[INFO] 저장 경로: {OUTPUT_DIR.resolve()}')

In [3]:
# 실행
orders, features = load_data()
filtered_orders = filter_high_value_orders(orders, features)
samples = build_time_aware_samples(filtered_orders)
samples = assign_splits(samples)
sequence_data, sequence_feature_names = build_sequence_arrays(filtered_orders, samples)
save_outputs(samples, sequence_data, sequence_feature_names)

summary_frame = pd.DataFrame({
'split': ['train', 'val', 'test'],
'count': [
    int((samples['split'] == 'train').sum()),
    int((samples['split'] == 'val').sum()),
    int((samples['split'] == 'test').sum()),
],
'positive_ratio': [
    float(samples.loc[samples['split'] == split_name, 'label'].mean())
    for split_name in ['train', 'val', 'test']
],
})
summary_frame


[INFO] 데이터 로드
[INFO] orders 컬럼: ['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
[INFO] feature 컬럼: ['user_id', 'total_orders', 'avg_days_between_orders', 'max_order_number', 'churn']



[INFO] 고빈도 고객 선택
[INFO] total_orders 80분위수: 55.00
[INFO] 고빈도 고객 수: 8778
[INFO] 주문 수: 650566

[INFO] 시점 기준 샘플 생성


[INFO] 샘플 수: 606676

[INFO] train/val/test 분할


split
train    415842
val       95417
test      95417

[INFO] LSTM 시퀀스 생성


[INFO] train shape: (415842, 5, 9)
[INFO] val shape: (95417, 5, 9)
[INFO] test shape: (95417, 5, 9)

[INFO] 전처리 결과 저장


[INFO] 저장 경로: /Users/jsh/Desktop/class/3-1/딥러닝응용/고빈도 고객 재구매 지연 위험 예측/outputs


,split,count,positive_ratio
0,train,415842,0.018519
1,val,95417,0.015521
2,test,95417,0.023235
